# Upload the seven trained Qwen2.5-3B checkpoints to Hugging Face

**CPU Colab is sufficient.** Run cells in order with a **write-enabled `HF_TOKEN`**
in Colab Secrets. Colab mounts Google Drive normally; no rclone or separate Google
OAuth client setup. Uploads are enabled in the last cell.

Seven **private model repositories** under `Seungjun` by default:
base + RL step 300, Kimi SFT epoch 5, rejection SFT epoch 4, 7B-distillation SFT
epoch 5, rejection SFT epoch 4 + RL step 300, distillation SFT epoch 5 + RL step
300, and instruct + RL step 300. Original B0 and I0 already exist on HF.

Copies the selected full weights, saved tokenizer/chat template, EOS/PAD and
generation settings, checkpoint marker, and original training manifest.
**No conversion, quantization, merging, tokenizer modification or training.**
Optimizer/scheduler states and other epochs are excluded. Weights are read
directly from Drive; the notebook does not duplicate seven models on Colab disk.

Each upload is checked against Hub file checksums. Completed models are reused
on rerun; failed uploads can be retried. After all seven succeed, a small
`transfer_catalog.json` is published in the base + RL model repository. It pins
each checkpoint's exact commit and lets RunPod download everything automatically.
Source Drive files are not modified. Existing unrelated repositories or different
checkpoint contents are rejected; use a different repository name in that case.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
EXPERIMENT_ROOT = "/content/drive/MyDrive/LG-AIME-Qwen25-3B-Experiments"
INSTRUCT_RL_ROOT = "/content/drive/MyDrive/LG-AIME-DAPO-Qwen25-3B-MiniBatch300-MemoryFix"
HF_OWNER = "Seungjun"
HF_PRIVATE = True
RECEIPT_DIR = str(Path(EXPERIMENT_ROOT) / "hf-checkpoint-upload")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "huggingface-hub==0.36.2", "PyYAML==6.0.3"])
from google.colab import drive, userdata
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Enable a write-enabled HF_TOKEN in Colab Secrets")
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-qwen-checkpoint-upload")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAY29tbW9uL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAABjb21tb24vaW8ucHnVVUtv2zAMvudXEDrJa2psBbZDgOyybodh6AbscemKQI7pRY0jGZLcJCj630dKTuLEwbDDephOJmk+vk8kJYS4bp0qaoSPXz/fQKVr9KBMCapp0JSX1tRbuLetM6r2sNZhAQ7n9gGTkzYBnWubgCUEpWufCyFGI71qrAuwUH5R62In3ntrdt/WjypnV9CowL9Ap/5C4mg0KrGCUv9CH+SDqlvM4PIt+OAmI6DjMFA9u+i5X6ir129kNPHhNHnZrhqfnMfgKfRsiVs//eZYRuNbhzPl51pPPxAwzHI0c1uizGKYLF/gpisg6+pZOx1wxsElFz2GVFkqiTUwjeVHa7bX5o1yaEK+WpbaySTs69hoH2Z2GcXkEpCJUG5L0aI7Uz7zbVXpTYycp2+4AJGHVSOSW7yYvW9u6e6kWIsMlIdqMqRmx0x1jowxXWtJZU6vsr1nlUf8Uvw0oq+t6tYv5EFjfV75rZlLMlEvGSuzE2C5w6ZWc+x4OiW37th1du3/L3Ir67hqIi8WfzD0+es1J/10thU5+/OwnGb6lOZ/zXKkK/olptSQqecnIwF2qI7hviDE2CjtZrytuhx/wq8rMDYkOBGrl9kBSLeJbu/SXqJbpwidtKJ1QpJwF4XgML28gJSWLIU4Rxg7njK2XhAyYI6P28oHRVtzSqwErOseKXxqbTDamAYWTuwdNrYcR+VTkNPySBvcdvgbN/w07ZXaqtJLjnacBjdzbAKk2+ZX5hp51b53zroxfDeapZ4uG2Y5oY/fJ648IZOvsr2Gs9MqLz1zKovYOsNwfIgx15q5ojaMJGZn/xqyEEErTff3g1dorFhW4p2Nz2B8RT+BClBsA8JjDP00gUe+3icahPjo3ViDR0G5c/I0nTwJg2s6B/7voFa5R1zKl2O4GiLczWFxGLGuo7mgboZqoohGvNjOdJkG4+gVJoiOa74VuhR32SR2RH8Vno5gf/7i0nga/QZQSwMEFAAAAAgAAAAhXAAAAAACAAAAAAAAABAAAABldmFsL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXATtxE+cCQAA8hwAABcAAABldmFsL3RyYW5zZmVyX21vZGVscy5weaVZWXPbOBJ+16/AeB9AThT6mMlUShPvy1aytVU7yUyO2QeViwVToIWIBDkEJdvr+L9vNxogwcPy7K6qbIk4Gn18faB5cnLySRYya1Wll6xuqoPUQmeSZVuZ7QwTesNUWe5bcV1IpjZSt6pV0rC8apgoCqaVluy3W6nZQTRK6NYkJycni4Uq66pp2VaYbaGu/eNXU2n/27TNPmsXeVOVrBYtLmNu6ld4XNBMVpVlpRNV+bmNupGmXbLbRrUytQQXHz98+Mwu7bYoTXNVyDSNk0aaqjjIKE5q0QDjZn1+tVgsNjJndgnyFuHJ8WrB4LMFEo7fxGzFxaufothO3Kp2S8Tt6qSqpY54c81jJgzKIUVJJPCDmsm2e71jSjPgsYkKUV5vxMqtBL7EJnrNvmfnZxc/uq94ya45j3sqlqFkX29EKyNLjnhpZLtvNExt5R2pApgkoW6lutm2VnwTyqX0Rt6BbDjEThkvq40sEiNy2UptqsYkdkWCuuR2hxYlmPiSRTDbyk1kZBvhbFJUYmMiWo5ipK28AwbiNXeHl6LmV8lBFHvgIY7jgTxM5cRMooxlM4qZLIxk6ylP/Ir2whZdtcRRr51GKNj2Ox7ztmmqJuLvK2ZpOC0wMGCzMZyIoEmQAlpkRAnZ6HWDk4uAW5x16GHfuVVICziyU50cQ8MRe+9g5n3Vvqv2ekNc5vwXZYzSNwMuV+wBiTzyXlsWcfaE41jDT93IXKF9B/Aaah5EKYAOLY1RlNdDIrNKzfk/NPhfXYBRnuMYP0b9WxIf4NjJXtci20X8zW986ZiM12dXY75Ql2fsDW1+c8lKpaPzs7P0jP6WpAbTCsAZfKV23Uv2Ov4fBdiCemTzhAQ0CTIEaA/ViocHoJZ6g26yPqw5+KlIqzwHVwHsQqSxoNst2QFBR3QTiAYluAWKvUMj8DQtZSvs3pRfhchDtVjyQGVOA2hC9oK09oKV4i7C1bM4fEYhePpIHS7IrAOvmPqQj6Xg7AqjVGpTRl0p3droswT+2+Z+yXYS/mWVztWN48/SvQxDaher7fxf2Ns7kYEC7mrZqBLonMq6ysAlqgLtQ/EJMHVAxxRZZhMTpKN7Zm5FXaOHdenIhxE8CH0Z8sDLi9UVatAyYLlcc5yGoBOsOBJucv6vpoJDepEhLzWQRqvm3mrqAYR+BK0iVa9VCBYmb0Fud+IOYiG/YpeAAxinyFuKZieb1Coax1txI88DzVKQtoGUqNn4Ceirq8mqgOAQ0ZGza3BYPAjnBIK99nx41QHjSRciW9CKTpuqaocrzPqHMX8XBO2iuoGlwb5TFnEYNKck51QunITId9ozA/kLf5dCqxyS30BSGhrKChSmosExEQm/ttRcXXPPLSo8oTUPxqtm4Fmza2CvS8kPuxU4fhgBOoYnMaAj8BjHRyH3t86+p52sfjOETVOKNtuuCHq8E9TLeVNU16JITStrYtWh0A0Aq5FTPdZ8fpd1u8F6N3KcVfIOu/QUD+h9omfM0ktuoLTg6OcZFBq9IZZs3j4xGWhu6ihDX7Q/g/CHEQK2d/obqi3LEai9jSl4uQDdMUv1xqULba6GweDPlaYcaAENdK2P+8G0KbiD97Uw0lGFhULfR3CyIwR0dqT3wXk4aHGFoAIVgI5BWfB9UAZq+D9nGHC3pkUVkAgzxiEorMKMZFnzfkiQ8E/PZB3+6d1nq23rwR6p7jBZoDcOiKdULcAZLyBFcvjfqyV1UTM4PTgcS29c6m2MxoDcaFFvaNcPZ2fHVMQ//rNnFchZ9F7vN4DTMefAN6eQl5JToEksqnv6pto3mRyAiYaCbO9oWJwgWiBe1YXIZMQRKmBcNEVQqHSAcIXpDAJ7koOiIqKzu9TznUs9VlA3Rc6Fipv6/kDaYSXntD+lMo07Q/Z9BJqjZvOzJzlN0H0KH5Ocrj3GrBNrmgmm0oyi0XPAJ3dD+DtboasBwvpU7ezqY8lMlnbRhzLdTJ6+kVo2Ai/wT+zvF6TPkGqrndRQUT5RL/Bu/jlCmHfsjc6Gd3Lc9r6W3MZv/set1Bf8mBe+9dEaWwsXTDTZFhJnBmWpDO50UGxjGDM2FtpTloE6gowgK6zT/XKXdCqTWoHAoIF3+cy0Hi64shU5nBMhLRshMaBjrwRHlqxQpvXXWRh4Fhs5/yQOIODbD5+gashz2RhmOx7tVlKOgkkr0zA3kSidJUayDBbBf5oGa8G1GrA2x/hGZZ5xeB6mOKLD33xTZQr3i29/5U/rB3Lc+avzn3585ZLbm2+wo8oRFrCvy/pAMQwC8PynE3eHTtTZOGlBWQVRE64hwxs9z7aiTf1k8lXpr4IPNqA2w70hmpHj4aS8Azsb37gYmWFwVphMO437SYeXXvSAmwcIYJRjVwx+dvQo8VsQ+rHHo0dYy84e0enDsg23OLEvWj7ouMxShAom9g2QjgYMqjo6Xnf4voexmHcHMtRXR2dSueIh09DjY85M6+X5tsuQyuh+Nu2fhZdhygHuzksRLQeSsqkbvPFCYvDtthwLiHVQkgUcEi1/S6DybUXrXdUXFnOrMaXHeHJ/Dve6yIgCULiLpjIFa3x7kcrzGoFl73HYtITrLMG/Dpp1eDOoE7PPseEEqx84WQN4dp6Fv4iZx/4qnxX7DRRTdatK1P3px/d/Z67N9zOGOzAaZEcXYCEQSmxAK13v/e19pLbaXkJXYRc37kVwPUsrYfzom6NQUmIKpiLaRJS/lkGHwd5Ilyyo0t1ItW+BE/dgbLNcbi7fV1o6g/sxUDk6dTSuxqwau0XKMNxL8cOPhpDvVoJA2LRDM/qx2OZPGnUjjPJgMPDSPk+4OJZu6R0A22v1x16yna5utStJwCV9FxWQZrtdV13+xZIVNe6ODvItZoi5wnRckZIHzBTO7kAMhxw2gSvYNhIvxDWAa+VzED2i0xDquglLd+RM3VTnTj3DYI1RW5V8LLJONkZFTJnFF3GGw/MTzSTfMxnueAnXED6pSZ1Snr4xdtVugFnkZXjsnBzWsYddNv8hgPt4EsB93JYLWPR7+p44RQ6TNhKCuTpIKA0cLQtR+jleELylGBh+Wg/iKwUqct3RmdDoLVjaQK1kjzjIxr4VYmJaYPtP1w/7r9qW/z8swZIk7RCSGAqWczgIsJK668fKMd+j1vhXd3PG6171WHQipmCg2xE0zpxJ+6muxsF4H3bCuxVhwe+u4s+Wu7/YeNKzDMkf0uemL+R+ZnvMBEzLW/bhy+dfv3xO8cVe+Fqke+fX87LEw/slGKQSUddQe0bdhMsfOLf4D1BLAwQUAAAACAAAACFcImm7EzAJAAClFgAAHwAAAGV2YWwvcnVucG9kX3RyYW5zZmVyX3N0b3JhZ2UucHmVWFtv2zgWfvev4KYPkhJZdtpusc3UC6RtJhPMtMmm6WCxhiHQEmWzkUkNSTlxg/z3PYekZMlOOqiBNubtXL9z88HBwXUtrmROtJGKLhipFCtKvlgaQgXsspJlhq9ZTKRgwzu6IdmSZbeV5MKQTFacaVIouSLnUi5KRj4quJwcHBwMBnxVSWXINy1F812x5pte1oaX7aqeV0pmTOuBJVZRsyz5nPjjK1gOBucX78mEvDw8fDUevCBXJRNmQ2RBlozmSsKrQipytTFLKUYfvn48JXdLxkodk99+JXOq2YgLbVSdGXLHUEE4yeSq4iU1XIoYaObU0JgoejfSmVQsB3l1JYVmcBWtYRjKQ9WG5PJOlJLmOiGfJZGV4Sv+nSlS8BIMQhVztsmTwfXZl7PrP89SJ/2b8eDDb2cffr+6vPh8k/568cfN2fUX2J8OCHyCIzJayZyVh4mmBTNMaKl0EG8PuvsJFzm7T9C+Qdy+z6Qo+MLv2p0FE0xZHdP9QyNvmUDRDzs0dMUyTsvUHup0RavuE5rnLPdn3f21zOh8V5wVUwumE3Nv/K1sSU2KhgSzs+QbF9/oUyd6dNiTyAA2j9Mt9rqMc1rJp46GxNKYDQaDnBUOuKmuaMZCBFhMDsGtec7RNKDtfANsJ+MYvc7UmqULPp903BedWHHwKXgMMWnJRO1usrrNuYJNBdDUkxtVQ9iwe65NKm/t0t2tNQbaxMdAknN9m9q9Dj3F/qo5QnCyJyI56kpIDgnIZt/wwpFOCsUYedfScHJbqpRrRiDiAa/sTCmpwiJ4QK6PJ+Sh83iERE+S4+KRnPP3BPd+IYKBPNSQklFtSNBSfeIDVFsNdmlBlCq2olxwsWhSCEbXnVS31jvJj2kHN0tGXo3J+Xt4LQwQgsBDIxKuCcR4XRQ84+CChHwFbakLyo0GaJE7Ds77MXU6l7UhL8eOAQWBOOQZFNAlvS83l9en52fp9eXlDVqDA5/TjiQ/pm7lzKggcwbOYnlM5jXS0D515JIIiQlRrSGREp+ZqxF43FBlkqCBh6mVIA8BegZhEJyQXfdBBDQ+8Df6Lol7ggZGGkBYl5TdcXcffQzlmN4hjVSbFMJUhC6hxE31SCEPGwyglTRsEizs9ROIRVdHWD75DGXEBxIUCXcR/UaJoCvMuFkJNzwFyOASPIhE8c6nTae8eMCjsRRoXZflippsGapgejr8Hx1+Hw/fpsPZEXJ31KLdOPiTlrWPgsAh5SkZdJ0tCdXEK+MghA6zYgEAGrG8a27ZRkPUlhD23jzTwOZuHcwiFLmxBaqE5iBwxNrdrmKWFIQLlLpQMxPiOorIPyZ2x67wuD0iQ/t9j+uPFP9i+ZJa8L9qRm4FFDZiH1ruDdxQVZ/zuq6OEsClLNcsjAAnDUOXEAEeWNhmdoVBD/QgPC3ZrUAQplBOJ2RX5ilcm7W3wB4BZqnAmgWI2GcnPQBjBHJRs3azm6jt/amjMYu6dFGqyYQETXOQqjLoE1YMO4Q1Zmyb5Zs1lMAwwHoL9diMLDpGnzZdKFgFwbnP0rOyBX+cD08vPp0N/3PHxMt/Dl+9H57dV0xBioYyEqBlkW9Lo4Q6W8Jj65RRS26rlE51ge7ySt9CnxDMrJKwH3Tsg3Uq9b5tyCS+fE1foXU8LYvQl1t/rKjgBWQka5ItlREJg1IutC/XwS4Fe2gV2uGWYOAhglQt0oa4q+NbcQFQCa0qJvLwIQC3QZ6C/yHLaVmrjNn8ZiP2aEue6rSSmt+HEdzLgSgXthOCy+Du0Joyip/O2EEjSLrPoJWxw+DvqOyz90ZrbkSPveSOCvu0C7XBMJVStdChj2Z/aQp7NQLF1dUaWnAIjyd6TDxu78KVaTAcOrKYIOHdrO2TMLtDZwxi5TrkUDZjnxIngfsb7MjQaj5152BrJAKE8fW0cdCsWXctAZuHPfU6dgQRbdOm65Xt54ZGUaEL5nri127P3vBb/3JbAD9j16/GkEriZ8QzshVw19OzvYMdkbuSeQZ987XdqP5hjbQ9aK847tkavSOgsZhsh6QEdppu1KXZn6nLW4YObi/IKfQkNm9ZlLB7miFEwM5MZAxRAuUJZgQNTQtcgAUGIXZv2EZahXHhp6rEEl1xrWHTd6wTMm7LABoWAYhydgoT03Vp85BVNty6Cjk/gSTwgO/z+/CZPROH8LFO8W05NHaAXZZCr1fVxm8adm86fbqTC02WohAgHHJM7NwXOoETbXIgEU0Dq2gw28vSNsU/AfstixfkUpQwV0toy30Bbo3vusJf9iZM6CCxEpbyDrbMEjoJ6FpFnmyzJY4IAg0KEA2rBEMijBKEOKqCjqjQC1bMRC1KOQ+Dw8A2KFUCSRv5QFXHvjfsWbSyyTqx7ajGZih0RTtwl/0xAMQf9kbZKNrq3YfI0QTy4H04jnsmHzaK+ClLQURBz7L7WwU2ed3xrg/7vRmvxxkqQ1HWetlx/PMw9QJ8AMiftKDEWoSInPw7eCbD7TJomPgsi3yez7rRTvvgAsRfjTuofq56LKo65WINVpRqE9pVrj1V32kek3euqWxOcf2623s2B2372adjOe81lx+WEht4bHKNBI1rO6iBZSDBnF99JRcf9ejrV/i/HWt8FujnunAaiDXPOR3qFXcpHppVtRmCDBP7K0hc1zyPEXnxCuCjNm54iW1eVCn807z9XeLZD5ZEqWCOmGR6HQuJvywxBV+gOzZYSX46hfgfsICcXWZU5ZgNpznPTPidV6CZlR+VQhXwL2qBf50i6QomMmxerCbB8+ltuoaYVEAystBaI6iUvJthEVDAzknlDmHfYk6voZtFHfvpLNFVyU0J4ywk1MjltHZq6fX04AAk5LGwhYKdxWzin6LODuz4BblaI/CieRziRmMHtLFdWmvMot4MgLDzlC0Mj3ciYxd/RYAgewA+jzhsIdDXXPN5aeHYgdQ2Kq2Qk0b+6bg/g1zf/JdcXV+SN+PxuJ1FnLzWazOMmAIKhPFKdXwIw9878nZMDsnx+OXrv5UcRwBrb0revsGfIrq8Y7IA3g+WxwFyPpjhDzhuveUJu+QTf99Rr/Fi00jji6hJBGjeh671e25rnj7+VAbwUU78nIh2z2tAV0ahj66WG82xTOZszSHY+z9utBPx/wFQSwMEFAAAAAgAAAAhXKGlEXVeEgAA8TwAABQAAABldmFsL3RyYW5zZmVyX2h1Yi5webVb23LjRpJ951eUOQ8AZBCU2/ZuLB3cGY1amu7YbUmrVnseaAYMEgURFgjAuKhbo+DrfsB+4n7JZmZdULiQotw7DIdbBKqyqvJy8lLJ8Xh8fvNpkqXJE3tbxI+c/e9//w97d8m2WcgTlterJF4HVZylLEhDlsdpykN8j8Nu6/QmC9l6w9cPeRanFauKIC0jXnjj8Xg0ird5VlTstzJL1d8FV39VfJtHccJHUZFtWR5UmyReMfnyBr7q+U/BNhmJYetsu81SL87UwDC+52Xlss9FXHGfVqKB/DFIvKJO8yz01ab8ssqK4J6rubRvv8yDNTcm6dHEgVINxq36m6DcuAxGxWEAyzUHhw3w+H5T+TisHI0ur28/nN2xObN+/8zTiaI5aWZMHr+zRudnd2f/ef03f+/woAqS7J7Gvr34eH77/ubu+hbHNYT8Ok+yIPTw7Nbo7vbs/ZX/4ezq/eXFR6IIxOI0Tu+nwA1/G6RxBAyTo99fXV7cXlydX/hXZx8uPsLw5xGDj7XO0ii+F6NcZt3zlBekBX7nDTHJK4OIVzwts6L04jTkX+R7QazKHnga/wO0Qs3ST7rkypyv4yDxaUAJ2831qyAMeShftMg/Zutg1WyIF6ARXvWlwm/rTVD5qGkJyMv7LU5/C1rre7R/RamsQDu+M8SqqYZBnvWfj3aj0SjkEYtTkBZP19xPgy238X/OjGiiXgNbUZ/FY3pa8KouUmavsiyxcYiXB0VVOmRjaVbRNC8u/WBVZkldcVu8sjzPovdxyppp9MomwvjBZXBAV7hZAet6UZ0k26Bab+xCCM/+82zyS/jtJIvwH+fPv5jShLPTrllc0rpXWcr1QkDQ2PzidMnm8w7LS0vABo4q6yiKv9AYKQnHcSQDDXXGsbZQC5c98CeX8S85L+Ith5dFloGtxWlZFfUavib0RPIaBhRPwGwxeSGOV1rLBVBZtvhO4uhRYVPTrmDrUyE3oruwcGPW0sHTwhOP2NyaMfn+9NTS3MFPHOEJ6MjGahaDbXFJu3023EN7OckgZcXaggWf8H+umCGZAEpTRhVwQZJ5AHu0hGTguWWyQckORpaL73GzcjJt780S9mJbAD/lVBiG1R1CLy3cs0FKM2YAb8RZBGD5oCCpFvSmXh0vaALrTX1/DwyJAL19mO3VVdygtYbogoPhxuGI5oEsSuAM6gOLQHuFdpHWgL101cYDj7ItwfLg1EIaOIqGC22C5yWvbFh8YeE6ZQz+JQadXzrsmzm9wxUdNJSEp/aewR5stuawkEPTcCRNm2lVAtkDt3/GYRdFkRW29SHIWZAksMYjT4VygFt+DIo4AGGyKpNvwris4nRdSXfeWthRh6A9SVfjw9KgLhJjiGGw+9YIdHEoYxj2jfAvwl/K90LUhzb/PiXxmHGDnIs8g/3eq82hiqDABL+lyNTGjCXgWDBsgLeN6eOnqxU2/uvo1yjnqcZXfDdrWXP/JJ/ge/YZfOO0w98n9v6tOoThBr4S5voHaQhKKBjEPrmAeNdQoVgFfX7uFRzQGqI/8K9EyPGC0sezfLGdGcuJ9bn2OsV9kq1s68Qi28jRUSEt22mxiz6I/h3veHg1Z6eJ/IndbTioRMLXFUcyEFiwEFxTsQV1L9nnTbzesHITFGHJVjzJ0nvU/GoDzsrw1ZoeMhgp+SJWk0cnuNLnM8M4sbedqR/PerTysoKJ8AofeDwNy88xSNZqeVFnh6bSXf8F7YqsT6maAnPBzuTu5JFnsHvY4M7qSHTRjgEB+w97j0GVMVQXbNAL8hyOZj9bCA8zoU6WtCL4jn+5EiRnDLRWsA4e0Y7gGf3r9vUDP9bqqaJBZb0F7QBnU9kO/OOXEKI1oiESDVjuWqEU7lK6l4KDcyn4NoOwKchjV+6u4I9xCRGsyxSASYSVFBC2PHRMpQgPYKq3idC1+CHYOL6x5YnniiJ8q55yPrdkGKkXmevVho+896P2NtebdDw6UAUaYOt46RGQIno6eMqQl+sizgGMVGSQRoiSeC7aro8P7Pa0eZtLEIHzKgCkCeZ3RS2D1zKGnBAwGs0HXaRXqL3OGH4nidEfIDRcw1MzdhrGcbSrDQIHNttdSKXRDtgEejwvLKwoevfgU5u4WlqpHAYwgPGqCHzxiUcKZVjiwsIn1vJlS6SQBXaKW2PbuCxhdST8uUDUQSpgjsjL3ZQwwrRKvSMviTpGH6yrOkjgQM0AD6z7zY//0qAWRFrtSUCvewD27+zHU3bCvjt988PJyZtZT+sGzvRBniIJIGOa0MEINcEK95+ltWmdE/9/GUuj/kL1TRbKVdvSI1a9Rn7qhChDSoP6R20FqYaDVc4atdJlJy7Li/gRnLDLyP664dGBWFVFqefZdhtX17nMrc/CJk5V9iwDB1xzQeC7dOUXhb7wQKQz9FRlDUhmGxQPvIDZXx8uNNaJVm+BCQPzALFF4QJw3vQLmMTDl2HaA3hImUIcQuATV0hF7HvRfrw0nMnzroskpL0IN8SEvfgBEoOjW+9AZUEcM0ueVNJI6nJj4Fz73JrqAkcviQ1ke8K3dbwWckHo5sywEvxLxhNoMGsAdhmMvmwsUtnmWun4F4jt/ezB2PF+hNexPuGxpIG2JP88EK5HljAQhja6ihMQBqQVEYR0pVDud5f+ze37n8/uLn4C88oymB4wsi1EFxnRI1am/LMRIUtQQW6SKykbP0KCLfe4DziEUQqDMURh1kbbwQhAENsErkGAUq5Gyi8CieRFkOAaT6jjFYRWpeQI6KqM0cqp8pw/sZo4Mnh6WkWiH5q53GEnBFbZETg0LGEKh0meLYJol8tcj4h77JbjjsASShUurglmXIMo4M0GkKHaQHpFJV6yB7lJCLijScHFcWKAXLU8bBEWgtym4GCansnxLvgNyAU/WRKSr3tBPD0oJTevgMQF+xdPyHifd07XO8IyYobCSAfrH5QlYj6iXzdy90UdWQwUf9u9AKqlGo3QgByisfhuLY9QxMbf/VFFtD4Gj6AGKuuUe0kzhkkQSJbMD/MS0ALhyQzahu69Lo7s4uiZtIJaeVfkLpGMeTizFK0usLYjGpk6lWwCiOrdx4CjVRGvakwJdsfaoyxP8m0O6ATggUl5yFZP3WRQbbUwoGqvYa6Ckg+VE3Utzzpc4LOQAI5H6oZiKGQgH5LEqyIonig1BmchyyngXzlVX608zjmgH+TKwT29hzxg0tTjreEEg5YWNxgwCb9g1Tu4R++5oEuGN96PVC7HeAH+xauPmigadxTWcmegVBBukR22NZlMfkkt9i1dzFCe64f1NrfVuVwGSW+FlaRyfhkAJxwYK2b9kv6J4czB2GBhJQGk8RDNwnAcaw2cLbIu6yRh/yWOMPn+r6Z00W/IIhj79Rlo7n712MesLtYQnv/6LFcdi1XHctUxhkLjJQzdt6b1d4HpLtMXBwBSm0BcYmGdmzT/4vrj9ObsrTxcXchrswIAPMtjTqaxgodbjw2ucpUxvL0AJKbgC0YLC/y9hvPE/xDk4IS0CVw/xZsOdp0DSdwTwxgEDAlXzADxIUTee6TI+ljxfMZOTp5luDXGok6Q+CU8B26cnHjsIs8wQNZjCDXHHJ+OXTZGiwvyHC8IVwkfOzRneLHbGjySDOVQEmpNM8Q7KILI+itaI0kOVPgZVXq3tDdVlZez6dQIsL11NhWvnZ8I/7Iihnfgx7BwVNKOnw6sc2kmQiVutgFpVKc7WUsBEMzA7Qap0K120YVOYkAJ1oT0jad3xzH+B6t/GxeQyQDq2Dk43PjLXNz/GVcKAq8m4JgCDDLk8DYs6sfqokk/aHvG5o60GcGmhg8aBnv8tCZYtxdnbz9ceFtwrJ4gStUJARHtibrGeHLSis1N1zdj+7ZjrNQeZLzYtdYTsY4MgmWALZ4dEWLT/YUcPtfhSE9LMpWwlfNFP4WjzAoib4rr5yK1oEcQZeDhs9VvVNxxBtIXUeKSactAriS25oO3LIN7Po+sGxG8McK6plRqQHhbHEbYImh5WRx+TTSA3juruheLi9PJvwWTaPn8w+nOSPQPXmlcqGIQxKTxdltXCCrsh1OwhqCA5B+jGgp3pIh1xNWqAz73q5J64Mw4x0DsNxsI/HbtakCQJK+8scLDrzkQg1UKqhoQelPNYI7VqT9UJnDZu+gsj0emCIjuIQbfAF4B1kLEQzY7AewCDmMzh393/R8XV+07l6+4pMMM42dZcjBuqVYcFJ4b+YanMmFhruja6Fg2HWVO/1fC1TxUGGc80mMyvAgArGmqA3Rn4OpqJE/rLXKR23gul33Xqw1E1nkTTjzT9N30X2ei0rkYg5mBm6LmF/lEqhs8tYZLCHJfC6OEs2z4u7e+pPJ9utSSXxA2UW/QMl9ZV2a9pGq+51LPcFqNvzAFAOirAn7ZI6PaJORRBQWVNbbqRe2GF9U/4mPFqOzZYTdIdNT4kkyZ1hKiTrK1rKKa29xzwFHnbDRZJ51KmfbeKPZvS5fHFl/sffeooJpH1z/UByPOFzPpfTLGjEUd2CyvD1ci2gnbP9vFvsa97jlf3+OSkLsuteNOtTdVKawArkn/nnq4etPxqMfk/3vl842Wz0FIFxtGjy+tTWxdduxFAVAMrRaGkj0e6SP1hmb7NootVpIlM7WHXdfABsEDdSBPOHZlafDAUdIJqQKDch5GX44Wj8fOJYt0qcH6i9W66evgsQwS5FL6vrDMkkeu+hc0Dmd1ldeV9uJtwYE0DzWjcJSP7ka5wJ6Rq6y6zOo0JNG57FZXHFovhEs3u8aMbVCH0iYynEa76UKpHV3SU324c3umNMC46xQX+uYNoxpO+kszjMsGVEwCR2z+0rqPfSBbSIssiUp6liqKiWm9KHBYq8/7fSEi2Q3NWur1p7ubT3f+7fX1nTVUV6q6aRIAzdHXn06rCktYod3ZHqMmvN3TDdMwlX9ZcxCB/QqVoNSPFGqIcZiutobbrUGkoda7y6FmmzoNHgEhKM4OKnkP5mG4SYZJpeXzDN73cbCfOkuzxlCUr7LsAXKZAhtzUSFQaCrKFKlwsF4D6IpuDazCmT7Wg2RXtOOSQRjq+HKmAUJ4Ub2aREO1M2stIwBvgftxmClkvwcJTc2RhqFGishBtW/BKg+iycllj8d3pD3qexFbKR1Vt2XQRb6k02gMa7SGDkRhoh69JxBr++uCWt36BGWpnijJMx7MACWaszDjopxLQjZuOHR+q/pZTLcgxWY2H8gtGQxU3SOz16mVunFVcPayjtE9iRKsYTjbuqwEiJlZbieXNXynuKZt+cZBJ6Zu3Q3fUGpPJvvepSs7Imc1Xd/+REOFA8IeTlzjgYRT4RRcLTjKeGUKLP9+RR5s5LyykxOCnwHtFI2ZqmtMNoFQRV49NfNm1eOoOjSp97LdialUXHyZCHXvrnpIsz/SugC48e81Zw8pCEsWM5CklHoDA2aDqm7DbNn8YP18aR5LUuvdfV/jLz10SRTrpFMlfyZ/8aC49BNeKA3osNI1cMMckkCvBZikm7Jh8xWpPaHjEXFYW+e6CifiMl3GcHWnDvaQuuxUI4RkbI9JIqschOqhfnL8tFojBmIDJNjq1mgeKThp3QOayGT3K1IikhIU+jWsJXK6edx3BMIB9Eua/VmY2tIU5NXAe+ryEInK3n77zhKdH6fIRtuBHouX8DWy/k59V6ZSNlenohr6z+u+VTXtRXPIr+hpUzaLeT8ws8Mj9YOQ/b8weYFRRg+peVEGu5mxwe4u2HQF2FCJnPYPdI7q08x7pyEQJmlMaUhrYdEUPme2sYOmtZg4Yb7ptInO+119w54L6TQtOQZFp0ND9pb1ugtQHLTXPv8V3Hz7wmZEnV821dptxXHNU8ofmxlZmdmOa8tGMUQDUfKQbZtOK4t+q/CafuqGybLxyze7HRwEYRjjwkHiU1fuXB7J6SfTRn11/0aOrbrqPaqa6zN6Xxzv7KRBs79gikLIN1bYub/s2rfIIcbqS5eePe6RMDb8xGnd1lzzAk40D5tMFb8rCoDnU+NiT12W46+M4HT7o6wjrKljwJKVFEoc0co84KC6dbs+0DQJc9+ZtZo4XVGcxeLPHBvDm7tJF2UEgKa2NDetW1UvnJ75NcfrQsBAYy96HcPY9VTnmD5SOmgfUbWqDvWSoqIOd83ulWN/UZMR8kdV2wdgnC1/rEWqPtgBaH4qdcvM5i2KmID74sd4tkX1Ca/a5tYBAvKSlwDBYKIoG4mnzqHpoFAJQo0Jts2l41c3p6qgU//ugIzfd/E/Qcmnn64AmCxH/wdQSwMEFAAAAAgAAAAhXL+WsdmJCgAABiMAAB8AAABjb25maWdzL3F3ZW5fdHJhbnNmZXJfZXZhbC55YW1sxVlZc9tGEn73r+BjUlu05j70Fucq19q12Wz2GTVHj4gIJGgAlK38+v0GlCKKohWmNnLkQ8Bc3V/fPbihYWz7zeXiw0faLKchbMZCw/KGv1r3mbrx8tViEcNI9fdi0YVI3eXiDZvfBtr2l4t/Y+NF/U+81kv55m7mpt0fK0OIvHCvFfmUuKLoNfNBcF086RSYE8IGH+dt1I/N1F/Tpmnz5YJrbpS8o98M3WMWlj+/m9+3YVpdzkuWQ7fchKm9oSUOukgrStfbvt1M40UO276pCIVuZGy2AwFpu6Hc1I0HS5eS7bFdtxuwMHTzyzjRFlDupj7D5XW7bpuxTEdsjnzx1T8xtRyn246+Xvznh1+O+cam5Y1Yjqt+mB6xPU7hivgF5g+Yn6Ux8qbSu4AG0qrRByxj8Z7LOnO50IcA+GcB7JfRpy2lCVJpM22mdrrFfqNUlNykqLzEIwVvXCTtnOUxORtMDMSsKJpJ5mMqMXAXeCJD2loqr6ox/IpTYQ0npPP73GIM623Xbq6qgBZfzdwv1NdPVHy/fnm/vkrvXKFRV5oBy/dSU5+XmjqQmmXGPaf33I5T23UnwNk395NhhniATD9BZuPycPGfgGVjc7fzz5kD59Y9a9EPinvifJ/R2z8WJ7zypMrgrM956APpmXHVPBbHOW66DQOMuHmk0T9Q4ROYpzR4CuMJ5f0BwAel7TnU/zdC/RzCdjNOwy49ttC3z4Tx5du7HUfxPARHVmhpvZcxeE/G5xJCCdpTZlxHbjVTmT8XaO6ZOZb328cx/SL1mwkAL/KAmH7x/va7+fe7H5ffvH3//fK7b37613JmeOb3fbtp34QprSCd5Xta98PtD+2n51Tw14R+/eqKNjTMeq9oJlpv6+tuoMsFe133Tf22qf42v4xE2KoEHtfhU7Ohj/sTRwwyb+6G5+TbdFQzqLBzAJpX79aIYh/GOoqhq+2uWc9Qmx1Mqf1tzwXI+mrV0+0WPMTS9WHiZuZtM/ZDA7sJXYfzx/Y3LKi6esDQQOqlvYKmd1Nf9wy7EZoClYkwl7GhhG6kV/SJ0u4edWVuS5sM70Z27WNH63smEbim3dhAwjTcBBAlUMh1lr0a+92QaC4ywpAuD8yxMrgJ7UVoRYOpIysUnGUmTAnFe62NlqE4z/GOUa6sDrIwprSct5W2HrUG49/8/O3y29V89BVdTDROS1Z/ln2ZH/hriObDjvZGP66C0OZyYURhUmhLVtpAWRhZtE4mxOJssjEyJlxQCpkwZxGEociF9RSVSDxz8nve+4/jHHJnxa+7XVPJHyL+5e2P3/+8fBfixfv37/67/Gnoj0BH7jwla3QIlrLVyYVUKCiRi+QlB7K8eBvpCHQOUzgbKyOhAvfcC55EgQQ1sjm8PbgowHzMXEIOlqusqHBInnwMzCROKfqQ3SFWweTvYKH4Nv/uI18E8gPJc4Bz6ZWQSUgrBCnHkmM2pFkWcMtgOUslEcrXYJWCXo3iqTjEQgmRMXsI3FY3bwuBgUOwV31/1dHF2x++x8QRTm9Myg5Fss4mGmDM1kgXk2UueicMT0E7cuoI554IXGu7m5oK+vWvY7/pHltvcBqFXwouEddUitWQJnNBBlWKlzxZElaS1ZiNUSblBWFPyU44V8QBMK1qqIhxdRH7vqOA5PwJpfRYMYyHULvddUDA2Fxg7TFSJ2J0HgC9liJAn8aaYHhJSsBxAlGCf3l/hPQEwbNtOulMMuskWaGUiUA8G6lcUSVKoYpBVeu0t0U48tkwmLZRlpjJGInCHkhAaHYngRR2I0T/6y5f0RpJ6mXhH1M7G7t1NjILGwr4oRy0wD/rtKg0AxgJXJpsEUsZcySL8toJL6VTXnmI7dCfnb3DjsIlrGN7tdsnjA/hZcE/IXc2eq+K0FEjWpEyxkrOMSA8Efw5OOgWbYu1tiQIpKhoeUqRCY/IYnRkmU5qPt+m66YLG/BzRS9s9o9pnR/Fa3DiCo22DAz1mGXSg74wtmjHsmQSUkHV5oUIViSP4JKKBWDhdeX0JO7SD2vYIJJ/F1L70siPqZ2NPSfvihRJJGSsTImisTVRWJ1TNpkYylaXmEF0F44VJYvNCL1WllRjfjqJfYVKaohhepy+/nrUD3TOxktCcq9NZloVGeHTxiknEN7IJ+Z8VlxIn5IOHFU6EhiPiP2mpJiZsiqdjm5df9UmiD9T3u3bsILSu+lj7cpeWPPP0z5bLorLnHnwIUL7pQSDOsIZn8Sc0TUrzDAVYCBeMeUS4p/l0TGHOtJ5NDlnymWkG3QDf5NgHhE/WzIICxx0lJ7znkxMosLRWhfUBQR/KAgaMWQOlyFEhMTBC0o/JE6LuluHMyUzrQb620zmEfHzK33UYcYolgwKfMvgLWSMcCjr0VOQ59w7uA+hMIKcwJuPMfrsuFL4q505KZl1f9MSGqjUr5G6nxbBf704TlE8WwYxeTT40mWljawuVGAJCc0NFWULMVt4tMyTy1wjsFh0P0kQQiwLOaty2m+GXbtpKnMvbAgPdM7PF1Y7J4gQFgqigDIqqeRSNNmjJNIBPQ6qB2UNUifaD+AshN6DZY56KQh1Ei+KtbZezcz36fs7oYaGAb13pml/sfWygjiDgbMlVDLSCwJoRludGCsBTUJhQUaveVBIs1nnUNDAMAhLoe0VPqN1KZApcq0+bRHjJgzXL2wNexrnZ4yAtidZJAS0tNA/SdJCaUALBWVzUugHokdn7wrWKM1lkNxmwbK1EiZ0WCvPDeGMc9sP09jsNpmGcQrzDckLoz5B8fzKsRhrHZQrfIQqgxDGw9N98bpwFMXGOYlCAsaftBfSOKmiSFZYkSVCgTupaxhhuq43Q+NqV0pH+T4wf8Gq4jwezo+ScIUckSyCZDrKmNFnIVc44iYZtPCw/xoq0GekrGVMQhejfeUXzQhjp3Po53n8glXGmUycLSkhS72oY9IENGAqOkMoxQJKL4+uC7WWZEbpqCycqXgBx0M0saz2Zr5+G/yTkvqCVceZTJxvUzJrbUkJE5K26EdRqsIHI2mUGAwFm4IVoUiDv1lkp8giZGVFFMokJOvTmegjxaYvTffiDdsBobMRh5oyLNpTboT2tSL3wgb8sfVaVUmVNEKwlzXpAmotyLjTnvuEucL5acT9kJsRMfDFI+0hpfNvZJIREkkjcZKObOToTwV6FfTqMlooM6KwyFEj0zgXpAzEIopu1FasWMGOMQNMM4WxptLlqVsyjB5fHmHoyZVKHXt014CB4xYcQw/9KV6eb9ROLngUQk6ueOQ6WHGqlsXwQ6GHlzOKnbpqrgbqw4kEieHzMsSzC4/hnRkhsPLAe+rbgWHNGqZPqdtBSFXJkAE9Yf6K+jVNQ5tAKmznQ9a7bmrr56smDO20wjRmp489pjbhpr3CMXjcs9CkfreZ9kdtCSbQbuoXmybAtmJX1w0Uxn5TsYTY7+qGrh8eAFWwtIZg5w88MPpNAhPz1f+4uw/GMNAW265u69NubDew0WqfK1qD0aEOQ83b3UTDYoTF4QwM1c9F/bpNdSlYwy4a9pyuKHTTqj5gez/v78LHCj3Mw/20oqEiWrVdP/bb1e38cjvuD9uOt2m15+d/UEsDBBQAAAAIAAAAIVyhDlteyQAAABwCAAAeAAAAY29uZmlncy9xd2VuX3RyYW5zZmVyX2h1Yi55YW1sfZFbjsIwDEX/u4pswLzRSNnGLKBKiwsuISm2W8TuSYHykGh/43OufZUOWSgGa84XDKDsglTIcGgL6JZZw9Q5RWuUW8xKp87HfX7EqzWFE8zZvx4r8hjc6c4+QvLnaFZLDBljE4U0MqHYzAy+Nf/Yhn3dhnl/wWoL6wL6GbAHUWzWi0Wij3SiXCodx2UJPQQJgrSqPGyTxlhjqanftPvC3vYm2TsSJe+n3b8CntzY6smSPzZ/Nx+OmEr5ecN3DAVJf1jqWM4w/7RuUEsBAhQDFAAAAAgAAAAhXAAAAAACAAAAAAAAABIAAAAAAAAAAAAAAIABAAAAAGNvbW1vbi9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAAAIVx0Lccl3AIAAIYIAAAMAAAAAAAAAAAAAACAATIAAABjb21tb24vaW8ucHlQSwECFAMUAAAACAAAACFcAAAAAAIAAAAAAAAAEAAAAAAAAAAAAAAAgAE4AwAAZXZhbC9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAAAIVwE7cRPnAkAAPIcAAAXAAAAAAAAAAAAAACAAWgDAABldmFsL3RyYW5zZmVyX21vZGVscy5weVBLAQIUAxQAAAAIAAAAIVwiabsTMAkAAKUWAAAfAAAAAAAAAAAAAACAATkNAABldmFsL3J1bnBvZF90cmFuc2Zlcl9zdG9yYWdlLnB5UEsBAhQDFAAAAAgAAAAhXKGlEXVeEgAA8TwAABQAAAAAAAAAAAAAAIABphYAAGV2YWwvdHJhbnNmZXJfaHViLnB5UEsBAhQDFAAAAAgAAAAhXL+WsdmJCgAABiMAAB8AAAAAAAAAAAAAAIABNikAAGNvbmZpZ3MvcXdlbl90cmFuc2Zlcl9ldmFsLnlhbWxQSwECFAMUAAAACAAAACFcoQ5bXskAAAAcAgAAHgAAAAAAAAAAAAAAgAH8MwAAY29uZmlncy9xd2VuX3RyYW5zZmVyX2h1Yi55YW1sUEsFBgAAAAAIAAgAJQIAAAE1AAAAAA=='
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

## Validate all seven checkpoints and show upload destinations

Checks saved epoch/step, source model, run identity, weight shards, tokenizer
and EOS before creating any HF repository. These are the same checks used by
the evaluation notebook. Edit `hub["repositories"]` here if you want different
repository names; the displayed catalog repository is the only value RunPod needs.
HF_PRIVATE controls new repositories; an existing repository must already match.

In [ ]:
import yaml
from eval.transfer_hub import upload_plan, upload_all
config = yaml.safe_load((CODE_ROOT / "configs/qwen_transfer_eval.yaml").read_text())
hub = yaml.safe_load((CODE_ROOT / "configs/qwen_transfer_hub.yaml").read_text())
hub["repositories"] = {key: HF_OWNER + "/" + repo.split("/", 1)[1]
                       for key, repo in hub["repositories"].items()}
hub["private"] = HF_PRIVATE
plan = upload_plan(config, hub, EXPERIMENT_ROOT, INSTRUCT_RL_ROOT)
for item in plan:
    print(f"{item['key']}: {item['bytes'] / 2**30:.2f} GiB → {item['repo_id']}")
    print("  source:", item["path"])
print("Total:", round(sum(item["bytes"] for item in plan) / 2**30, 2), "GiB")
print("RunPod HF_CATALOG_REPO:", hub["repositories"][hub["catalog_key"]])
print("Visibility:", "private" if HF_PRIVATE else "public")
print("Upload receipts:", RECEIPT_DIR)

## Upload and verify

Hashing and transferring tens of GB can take time; leave the Colab session
connected. The upload library shows transfer progress. A receipt is written to
Drive after every verified model. A rerun verifies already-uploaded checkpoints
and continues; it does not start training or overwrite different checkpoints.

After **“All seven checkpoints verified”**, open the updated RunPod notebook
and run its setup/download cells. The default catalog repository already matches.
If you changed HF_OWNER or repository names, copy the printed HF_CATALOG_REPO
into RunPod. It resolves the catalog once and pins all seven model revisions.

In [ ]:
RUN_UPLOAD = True
if RUN_UPLOAD:
    receipt = upload_all(config, hub, EXPERIMENT_ROOT, INSTRUCT_RL_ROOT,
                         RECEIPT_DIR, token=HF_TOKEN)
    from IPython.display import Markdown, display
    table = "| Variant | Model repository | Pinned revision |\n| --- | --- | --- |\n"
    for key, record in receipt["catalog"]["models"].items():
        repo, revision = record["repo_id"], record["revision"]
        table += f"| {key} | [{repo}](https://huggingface.co/{repo}) | `{revision}` |\n"
    display(Markdown(table))
    print("HF_CATALOG_REPO =", repr(receipt["repo_id"]))
    print("HF_CATALOG_REVISION =", repr(receipt["revision"]), "# optional exact pin; main also works")
    print("Completion receipt:", Path(RECEIPT_DIR) / "upload_complete.json")
else:
    print("Set RUN_UPLOAD = True to publish the seven validated checkpoints.")